# Jean API en Kaggle: tu propia API de IA, gratis, con GPU

Este notebook convierte la GPU gratuita de Kaggle en **tu propia API compatible con OpenAI** con los modelos Jean
(o cualquier GGUF de Hugging Face, o uno tuyo). Funciona con [JeanCode](https://github.com/jeancodeeer/JeanCode),
los SDK de OpenAI, `curl` y cualquier cliente que acepte una URL base.

**Antes de pulsar Run All (una sola vez):**

1. **Verifica tu teléfono** en tu cuenta de Kaggle (Settings → Phone verification). Sin eso Kaggle no da GPU ni internet.
2. En este notebook, panel derecho → **Settings**:
   - **Accelerator:** `GPU T4 x2`
   - **Internet:** `On`
3. *(Opcional)* **Add-ons → Secrets → Add secret:** nombre `JEAN_API_KEY`, valor = una clave larga que elijas,
   para tener siempre la misma. Sin ella, el notebook crea una clave para cada sesión y te la muestra junto a la URL.
   *Opcional:* `HF_TOKEN` si vas a usar modelos privados de Hugging Face.

Después: **Run All**. En 3–6 minutos verás un recuadro con la **URL para JeanCode** y la clave.

La sesión dura hasta ~12 horas (límite de Kaggle) y tu cuota de GPU es de unas 30 horas a la semana; se renueva cada
semana. Para seguir después, vuelve a pulsar Run All (la URL cambia). Uso personal: no compartas tu URL ni tu clave.

## Qué modelos servir

Edita `MODELOS` si quieres otros. Separados por comas:

| Escribe | Qué sirve |
|---|---|
| `grey`, `marvel` | los modelos Jean (uno en cada GPU) |
| `fenix` | Jean Fénix, el grande (usa las dos GPU, solo) |
| `hf:usuario/repo/archivo.gguf` | cualquier GGUF de Hugging Face |
| `file:/kaggle/input/.../modelo.gguf` | un GGUF tuyo: súbelo como dataset y añádelo con **Add Input** |

También puedes cargar más modelos de Hugging Face mientras corre, desde el panel de Hugging Face de JeanCode.

In [ ]:
MODELOS = "grey,marvel"   # ver la tabla de arriba
CONTEXTO = 32768          # tokens de contexto por modelo
RAMA = "main"             # rama de Jean-API-Generator

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    os.environ["JEAN_API_KEY"] = secrets.get_secret("JEAN_API_KEY")   # clave fija: nunca se imprime
except Exception:
    pass  # sin secreto: el lanzador crea una clave para esta sesión y la muestra junto a la URL
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    pass

os.environ.update(JEAN_MODELS=MODELOS, JEAN_CTX=str(CONTEXTO))
home = "/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp"
repo = f"{home}/Jean-API-Generator"
for rama in dict.fromkeys([RAMA, "main", "kaggle"]):   # la primera que traiga el lanzador
    if os.path.isfile(f"{repo}/kaggle/launch.py"):
        break
    subprocess.run(["rm", "-rf", repo])
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", rama,
                    "https://github.com/jeancodeeer/Jean-API-Generator.git", repo])
subprocess.run(["python3", f"{repo}/kaggle/launch.py"])